In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import json

print("cleaned_data.csv upload karo (Module 2 ka output):")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


cleaned_data.csv upload karo (Module 2 ka output):


Saving cleaned_data.csv to cleaned_data.csv
Loaded: 11162 rows, 17 columns


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343.0,yes,no,unknown,5,may,1033,1,NaN,0.0,unknown,yes
1,56,admin.,married,secondary,no,45.0,no,no,unknown,5,may,1033,1,NaN,0.0,unknown,yes
2,41,technician,married,secondary,no,1270.0,yes,no,unknown,5,may,1033,1,NaN,0.0,unknown,yes
3,55,services,married,secondary,no,2476.0,yes,no,unknown,5,may,579,1,NaN,0.0,unknown,yes
4,54,admin.,married,tertiary,no,184.0,no,no,unknown,5,may,673,2,NaN,0.0,unknown,yes


In [ ]:
# ===== RULE-BASED VALIDATION ENGINE =====
rule_violations = []

def check_rule(column, rule_name, condition, severity):
    n = int(condition.sum())
    rule_violations.append({'column': column, 'rule': rule_name,
                            'severity': severity, 'violations': n})
    return n

# business rules (bank marketing data ke liye)
check_rule('age', 'age must be 18-100', (df['age'] < 18) | (df['age'] > 100), 'high')
check_rule('balance', 'balance must be >= 0', df['balance'] < 0, 'high')
check_rule('duration', 'call duration must be > 0', df['duration'] <= 0, 'medium')
check_rule('campaign', 'contact attempts must be >= 1', df['campaign'] < 1, 'high')
check_rule('day', 'day must be 1-31', (df['day'] < 1) | (df['day'] > 31), 'high')
check_rule('marital', 'marital must be married/single/divorced',
           ~df['marital'].isin(['married', 'single', 'divorced']), 'high')
check_rule('education', 'education must be valid level',
           ~df['education'].isin(['primary', 'secondary', 'tertiary', 'unknown']), 'medium')
check_rule('deposit', 'target must be yes/no',
           ~df['deposit'].isin(['yes', 'no']), 'high')

print('===== RULE-BASED VALIDATION =====')
total_violations = 0
for v in rule_violations:
    total_violations += v['violations']
    status = 'PASS' if v['violations'] == 0 else 'FAIL'
    print(f"  [{status}] {v['column']}: {v['rule']} -> {v['violations']} violations ({v['severity']})")

print(f"\nTotal rule violations: {total_violations}")


===== RULE-BASED VALIDATION =====
  [PASS] age: age must be 18-100 -> 0 violations (high)
  [PASS] balance: balance must be >= 0 -> 0 violations (high)
  [PASS] duration: call duration must be > 0 -> 0 violations (medium)
  [PASS] campaign: contact attempts must be >= 1 -> 0 violations (high)
  [PASS] day: day must be 1-31 -> 0 violations (high)
  [PASS] marital: marital must be married/single/divorced -> 0 violations (high)
  [PASS] education: education must be valid level -> 0 violations (medium)
  [PASS] deposit: target must be yes/no -> 0 violations (high)

Total rule violations: 0


In [4]:
# ===== AI ANOMALY DETECTION (Isolation Forest) =====
from sklearn.ensemble import IsolationForest

# numeric features (model ke liye NaN ko median se bharo)
feature_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X = df[feature_cols].copy()
X = X.fillna(X.median())

# Isolation Forest — 2% rows ko anomalous maan kar check karega
iso = IsolationForest(contamination=0.02, random_state=42)
df['anomaly_flag'] = iso.fit_predict(X)        # -1 = anomaly, 1 = normal
df['anomaly_score'] = iso.decision_function(X)  # jitna kam, utna ajeeb

anomalies = df[df['anomaly_flag'] == -1]

print('===== AI ANOMALY DETECTION (Isolation Forest) =====')
print(f'Rows analyzed: {len(df)}')
print(f'Anomalies detected: {len(anomalies)} ({round(len(anomalies)/len(df)*100, 2)}%)')
print()
print('Top 5 most anomalous records:')
cols_show = ['age', 'job', 'balance', 'duration', 'campaign', 'anomaly_score']
print(anomalies[cols_show].nsmallest(5, 'anomaly_score').to_string())

===== AI ANOMALY DETECTION (Isolation Forest) =====
Rows analyzed: 11162
Anomalies detected: 224 (2.01%)

Top 5 most anomalous records:
      age         job  balance  duration  campaign  anomaly_score
3663   74     retired   4327.0       301         5      -0.078462
4422   28  unemployed   4327.0       981         5      -0.069201
3775   82   housemaid   1381.0       844         5      -0.069196
4118   72     retired   4327.0      1033         5      -0.067829
9218   66  management   2149.0       151         6      -0.058530


In [5]:
# ===== DATASET HEALTH SCORE + VALIDATION REPORT =====
anomaly_count = int((df['anomaly_flag'] == -1).sum())
rules_passed = sum(1 for v in rule_violations if v['violations'] == 0)

health = round(100 - (total_violations / len(df) * 100)
                    - (anomaly_count / len(df) * 100), 2)

report = {
    'dataset': 'cleaned_data.csv',
    'rows': int(len(df)),
    'rule_validation': {
        'total_rules': len(rule_violations),
        'rules_passed': rules_passed,
        'total_violations': total_violations,
        'details': rule_violations
    },
    'ai_anomaly_detection': {
        'model': 'IsolationForest',
        'contamination': 0.02,
        'anomalies_found': anomaly_count,
        'anomaly_pct': round(anomaly_count / len(df) * 100, 2)
    },
    'dataset_health_score': health
}

with open('validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('===== MODULE 3 COMPLETE =====')
print(f'Rules: {rules_passed}/{len(rule_violations)} passed ({total_violations} violations)')
print(f'AI anomalies: {anomaly_count} (2%)')
print(f'Dataset Health Score: {health} / 100')
print()
print('Saved: validation_report.json')
files.download('validation_report.json')

===== MODULE 3 COMPLETE =====
Rules: 8/8 passed (0 violations)
AI anomalies: 224 (2%)
Dataset Health Score: 97.99 / 100

Saved: validation_report.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>